# Classify CICE fast ice with `shuga`

Notebook-cell version of the `classify.py` script.

This notebook replaces command-line arguments with editable variables and keeps the same high-level workflow:

1. configure run/classification/grid/path settings;
2. optionally persist grid assets;
3. optionally convert `iceh*.nc` to grouped Zarr;
4. run `CICEClassifier` for raw, binary-days, and rolling-mean masks.


In [1]:
import sys
from pathlib import Path
#####################################################################
# Make sure this reflects the correct location of mawsons-chest repo.
#####################################################################
repo_root = Path.home() / "AFIM" / "src" / "mawsons-chest"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from shuga import (ClassificationSpec,
                    CICEClassifier,
                    RunSpec,
                    CICEGridSpec,
                    ShugaPaths)
from shuga.core.logging import build_file_logger
from shuga.core.naming import normalize_method
from shuga.core.data_conversion import NC2Zarr
def comma_split(value: str | None) -> list[str]:
    if value is None:
        return []
    return [v.strip() for v in value.split(",") if v.strip()]

## run config

In [3]:
SIM_NAME       = "elps-min"
START_DATE     = "1993-01-01"
END_DATE       = "2023-12-31"
HEMISPHERE     = "SH"
PROJECT        = "gv90"
USER           = "da1339"
ICEH_FREQUENCY = "daily"  # "daily" or "hourly"
HOURLY_ROOT    = None        # e.g. Path.home() / "AFIM_archive" / SIM_NAME / "history" / "hourly"
# Time chunk size used for conversion/loading.
# If None, defaults to 31 for daily and 24 for hourly.
CHUNKS_TIME    = None

## classification config 

In [4]:
ICE_TYPE                 = "FI"      # classification workflow must be FI; PI is derived from FI
GRID_TYPE                = "Tb"      # 
ISPD_THRESH              = 5.0e-4
AICE_THRESH              = 0.15
METHODS                  = "raw,binary-days,rolling-mean"
BIN_WINDOW               = 11
BIN_MIN_DAYS             = 9
ROLL_WINDOW              = 15
OVERWRITE_CLASSIFICATION = True

## optional grid, path, logging configs

In [14]:
GRID_FILE               = "/g/data/gv90/da1339/grids/ACCESS-OM2-025_grid.nc"
KMT_FILE                = "/g/data/gv90/da1339/grids/ACCESS-OM2-025_kmt.nc"
BATHYMETRY_FILE         = "/g/data/gv90/da1339/grids/ACCESS-OM2-025_topog.nc"
F2_FILE                 = None
GRIDCPL_FILE            = None
ICE_IN_FILE             = None
PERSIST_GRID_ASSETS     = False
CICE_STORE              = None
STATIC_STORE            = Path.home() / "AFIM_archive" / "paper1" / "CICE_0p25_Bgrid_coords.zarr"
CLASSIFICATION_ROOT     = None
AFIM_OUTPUT_ROOT        = Path.home() / "AFIM_archive" / "paper1"
LOGS_ROOT               = Path.home() / "logs" / "paper1"
ARCHIVE_ROOT            = Path.home() / "AFIM_archive" / "paper1"
# NetCDF-to-Zarr conversion controls.
SKIP_HISTORY_CONVERSION = True
DAILY_ROOT              = None
NETCDF_ENGINE           = "scipy"
OVERWRITE_HISTORY       = False
OVERWRITE_STATIC        = True
DELETE_ORIGINAL         = False
LOG_LEVEL               = "INFO"

## classification runtime objects

In [15]:
if str(ICE_TYPE).strip().upper() != "FI":
    raise ValueError("The classification workflow must be run with ICE_TYPE='FI'. "
                     "It writes both FI and PI classification stores from the FI parent mask. "
                     "Use ICE_TYPE='PI' only in the metrics workflow.")
methods = [normalize_method(m) for m in comma_split(METHODS)]
run_cfg = RunSpec(sim_name       = SIM_NAME,
                  start_date     = START_DATE,
                  end_date       = END_DATE,
                  hemisphere     = HEMISPHERE,
                  project        = PROJECT,
                  user           = USER,
                  iceh_frequency = ICEH_FREQUENCY)
cls_cfg = ClassificationSpec(ice_type     = ICE_TYPE,
                             grid_type    = GRID_TYPE,
                             ispd_thresh  = ISPD_THRESH,
                             aice_thresh  = AICE_THRESH,
                             methods      = tuple(methods),
                             bin_window   = BIN_WINDOW,
                             bin_min_days = BIN_MIN_DAYS,
                             roll_window  = ROLL_WINDOW)
G_cice_cfg = CICEGridSpec(grid_file       = GRID_FILE,
                          kmt_file        = KMT_FILE,
                          bathymetry_file = BATHYMETRY_FILE,
                          f2_file         = F2_FILE,
                          gridcpl_file    = GRIDCPL_FILE,
                          ice_in_file     = ICE_IN_FILE)
pth_cfg = ShugaPaths(run_cfg             = run_cfg,
                     cls_cfg             = cls_cfg,
                     afim_output_root    = AFIM_OUTPUT_ROOT,
                     cice_store          = CICE_STORE,
                     static_store        = STATIC_STORE,
                     G_cice_cfg          = G_cice_cfg,
                     classification_root = CLASSIFICATION_ROOT,
                     logs_root           = LOGS_ROOT,
                     archive_root        = ARCHIVE_ROOT)
chunks_time = CHUNKS_TIME
if chunks_time is None:
    chunks_time = 24 if ICEH_FREQUENCY == "hourly" else 31
chunks = {"time": chunks_time}
print(run_cfg)
print(cls_cfg)
print(f"methods: {methods}")
print(f"chunks : {chunks}")

RunSpec(sim_name='elps-min', start_date='1993-01-01', end_date='2023-12-31', hemisphere='SH', project='gv90', user='da1339', iceh_frequency='daily')
ClassificationSpec(ice_type='FI', grid_type='Tb', ispd_thresh=0.0005, methods=('raw', 'binary-days', 'rolling-mean'), bin_window=11, bin_min_days=9, roll_window=15, speed_var_u='uvel', speed_var_v='vvel', uvelE_var='uvelE', uvelN_var='uvelN', vvelE_var='vvelE', vvelN_var='vvelN', aice_var='aice', aice_thresh=0.15, wrap_x=True, cgrid_combine='mean')
methods: ['raw', 'binary-days', 'rolling-mean']
chunks : {'time': 31}


## initialise logger and check that static grid is present (where shuga thinks it should be)

In [16]:
logger = build_file_logger("shuga.cls_cfg", pth_cfg.classification_log_path(), level = LOG_LEVEL)
logger.info("Logging to: %s", pth_cfg.classification_log_path())
print(f"Log file: {pth_cfg.classification_log_path()}")
static_store = pth_cfg.resolve_static_store()
if static_store is None:
    logger.warning("No CICE static store resolved. Classification may fail if TLON/TLAT "
                   "or other static fields have been stripped from grouped history zarr.")
    print("WARNING: no CICE static store resolved")
else:
    logger.info("Resolved universal CICE static store: %s", static_store)
    print(f"Resolved universal CICE static store: {static_store}")
print(f"Resolved classification root: {pth_cfg.classification_root_path}")

2026-06-25 13:34:54,379 - INFO - [shuga.cls_cfg.<module>:2] Logging to: /home/581/da1339/logs/paper1/classification/classify_elps-min_FI_Tb_ispd_thresh5e-4_BW11_BM9_roll15.log
2026-06-25 13:34:54,384 - INFO - [shuga.cls_cfg.<module>:10] Resolved universal CICE static store: /home/581/da1339/AFIM_archive/CICE_0p25_Cgrid_coords.zarr


Log file: /home/581/da1339/logs/paper1/classification/classify_elps-min_FI_Tb_ispd_thresh5e-4_BW11_BM9_roll15.log
Resolved universal CICE static store: /home/581/da1339/AFIM_archive/CICE_0p25_Cgrid_coords.zarr
Resolved classification root: /home/581/da1339/AFIM_archive/paper1/elps-min/zarr/SH/ispd_thresh_5.0e-4/FI/Tb


## persist explicit CICE grid assets

In [8]:
has_explicit_grid_assets = any(v is not None for v in (GRID_FILE, KMT_FILE, BATHYMETRY_FILE, F2_FILE, GRIDCPL_FILE, ICE_IN_FILE))
if PERSIST_GRID_ASSETS and has_explicit_grid_assets:
    cfg = pth_cfg.persist_cice_grid_assets(grid_spec = G_cice_cfg, overwrite = True)
    logger.info("Persisted CICE grid assets: %s", cfg)
    print(cfg)
elif PERSIST_GRID_ASSETS:
    logger.warning("PERSIST_GRID_ASSETS=True but no explicit grid assets were provided; leaving any existing config unchanged.")
    print("WARNING: PERSIST_GRID_ASSETS=True but no explicit grid assets were provided.")
else:
    print("Grid-asset persistence skipped.")

Grid-asset persistence skipped.


## resolve CICE grid

In [9]:
grid_assets = pth_cfg.resolve_cice_grid_assets()
if grid_assets is None:
    raise RuntimeError("resolve_cice_grid_assets() returned None")
logger.info("Resolved CICE grid file: %s", grid_assets["grid_file"])
logger.info("Resolved CICE KMT file : %s", grid_assets["kmt_file"])
print("Resolved CICE grid assets:")
for key, value in grid_assets.items():
    print(f"  {key}: {value}")

2026-06-25 13:32:19,687 - INFO - [shuga.cls_cfg.<module>:4] Resolved CICE grid file: /g/data/gv90/da1339/grids/ACCESS-OM2-025_grid.nc
2026-06-25 13:32:19,688 - INFO - [shuga.cls_cfg.<module>:5] Resolved CICE KMT file : /g/data/gv90/da1339/grids/ACCESS-OM2-025_kmt.nc


Resolved CICE grid assets:
  grid_file: /g/data/gv90/da1339/grids/ACCESS-OM2-025_grid.nc
  kmt_file: /g/data/gv90/da1339/grids/ACCESS-OM2-025_kmt.nc
  bathymetry_file: /g/data/gv90/da1339/grids/ACCESS-OM2-025_topog.nc
  f2_file: /g/data/gv90/da1339/coastal_drag/form_factors/combined.nc
  gridcpl_file: unknown_gridcpl_file
  ice_in_file: /home/581/da1339/AFIM_archive/paper1/elps-min/ice_in
  ice_diag_file: /home/581/da1339/AFIM_archive/paper1/elps-min/ice_diag.d


## netcdf to zarr conversion

In [10]:
if SKIP_HISTORY_CONVERSION:
    logger.info("SKIP_HISTORY_CONVERSION=True; using existing CICE Zarr/static stores.")
    logger.info("Resolved CICE store target: %s", pth_cfg.resolve_cice_store())
    logger.info("Resolved static store    : %s", pth_cfg.resolve_static_store())
    print("History conversion skipped.")
    print(f"Resolved CICE store : {pth_cfg.resolve_cice_store()}")
    print(f"Resolved static     : {pth_cfg.resolve_static_store()}")
else:
    converter = NC2Zarr(pth_cfg       = pth_cfg,
                        logger        = logger,
                        chunks        = chunks,
                        netcdf_engine = NETCDF_ENGINE)
    conv = converter.ensure_iceh_stores(dt0_str          = START_DATE,
                                        dtN_str          = END_DATE,
                                        daily_root       = DAILY_ROOT,
                                        hourly_root      = HOURLY_ROOT,
                                        overwrite        = OVERWRITE_HISTORY,
                                        overwrite_static = OVERWRITE_STATIC,
                                        delete_original  = DELETE_ORIGINAL)
    logger.info("Resolved/updated CICE store: %s", conv.cice_store)
    if conv.static_store is not None:
        logger.info("Resolved/updated static store: %s", conv.static_store)
    logger.info("nc2zarr summary: groups_scanned=%d groups_written=%d groups_rewritten=%d "
                "groups_skipped=%d source_files_seen=%d source_files_used=%d",
                conv.months_scanned, conv.months_written, conv.months_rewritten,
                conv.months_skipped, conv.daily_files_seen,conv.daily_files_used)
    print(conv)

2026-06-25 13:32:37,403 - INFO - [shuga.cls_cfg.<module>:2] SKIP_HISTORY_CONVERSION=True; using existing CICE Zarr/static stores.
2026-06-25 13:32:37,421 - INFO - [shuga.cls_cfg.<module>:3] Resolved CICE store target: /home/581/da1339/AFIM_archive/paper1/elps-min/zarr/iceh_daily.zarr
2026-06-25 13:32:37,423 - INFO - [shuga.cls_cfg.<module>:4] Resolved static store    : /home/581/da1339/AFIM_archive/CICE_0p25_Cgrid_coords.zarr


History conversion skipped.
Resolved CICE store : /home/581/da1339/AFIM_archive/paper1/elps-min/zarr/iceh_daily.zarr
Resolved static     : /home/581/da1339/AFIM_archive/CICE_0p25_Cgrid_coords.zarr


## RUN CLASSIFICATION

In [28]:
logger.info("Resolved classification root: %s", pth_cfg.classification_root_path)
runner = CICEClassifier(run_cfg = run_cfg,
                        cls_cfg = cls_cfg,
                        pth_cfg = pth_cfg,
                        chunks  = chunks,
                        logger  = logger)
outputs = runner.run_methods(methods = methods, overwrite = OVERWRITE_CLASSIFICATION)
for method, path in outputs.items():
    logger.info("Wrote %s classification: %s", method, path)
    print(f"Wrote {method}: {path}")

2026-06-07 11:21:43,037 - INFO - [shuga.cls_cfg.<module>:1] Resolved classification root: /home/581/da1339/AFIM_archive/LD-fsnow-red/zarr/SH/ispd_thresh_5.0e-4/FI/Tc
2026-06-07 11:21:43,045 - INFO - [shuga.cls_cfg.run_methods:656] Resolved classification root: /home/581/da1339/AFIM_archive/LD-fsnow-red/zarr/SH/ispd_thresh_5.0e-4/FI/Tc
2026-06-07 11:21:43,046 - INFO - [shuga.cls_cfg.run_methods:657] Classification speed reconstruction mode(s): Tc
2026-06-07 11:21:43,048 - INFO - [shuga.cls_cfg._load_cice:198] Resolved CICE store: /g/data/gv90/da1339/afim_output/LD-fsnow-red/zarr/iceh_daily.zarr
Requested static variable(s) missing from /home/581/da1339/AFIM_archive/CICE_0p25_Cgrid_coords.zarr: ['aice', 'uvelE', 'uvelN', 'vvelE', 'vvelN']
2026-06-07 11:21:52,644 - INFO - [shuga.cls_cfg.run_methods:661] Classifying method: raw
2026-06-07 11:21:52,647 - INFO - [shuga.cls_cfg.compute_tgrid_speed:187] Computing T-grid speed using Tc reconstruction from C-grid east/north components
2026-06-07

Wrote FI:raw: /home/581/da1339/AFIM_archive/LD-fsnow-red/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/raw/data.zarr
Wrote PI:raw: /home/581/da1339/AFIM_archive/LD-fsnow-red/zarr/SH/ispd_thresh_5.0e-4/PI/Tc/raw/data.zarr
Wrote FI:binary-days: /home/581/da1339/AFIM_archive/LD-fsnow-red/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/data.zarr
Wrote PI:binary-days: /home/581/da1339/AFIM_archive/LD-fsnow-red/zarr/SH/ispd_thresh_5.0e-4/PI/Tc/bin-win-11_bin-min-09/data.zarr
Wrote FI:rolling-mean: /home/581/da1339/AFIM_archive/LD-fsnow-red/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/roll-days-15/data.zarr
Wrote PI:rolling-mean: /home/581/da1339/AFIM_archive/LD-fsnow-red/zarr/SH/ispd_thresh_5.0e-4/PI/Tc/roll-days-15/data.zarr
Wrote SI: /home/581/da1339/AFIM_archive/LD-fsnow-red/zarr/SH/SI/data.zarr


## basic inspection of classification ... check masks

In [29]:
from shuga import load_classified
for method in methods:
    try:
        ds = load_classified(run_cfg        = run_cfg,
                             cls_cfg        = cls_cfg,
                             pth_cfg        = pth_cfg,
                             classification = method,
                             variables      = ["FI_mask", "FI_aice", "FI_ispd"],
                             chunks         = chunks)
        print(f"\n[{method}]")
        print(ds)
    except Exception as exc:
        print(f"Could not open classified output for {method}: {exc}")


[raw]
<xarray.Dataset> Size: 10GB
Dimensions:  (time: 729, nj: 1080, ni: 1440)
Coordinates:
  * time     (time) datetime64[ns] 6kB 1993-01-01 1993-01-02 ... 1994-12-30
Dimensions without coordinates: nj, ni
Data variables:
    FI_mask  (time, nj, ni) bool 1GB dask.array<chunksize=(31, 1080, 1440), meta=np.ndarray>
    FI_aice  (time, nj, ni) float32 5GB dask.array<chunksize=(31, 1080, 1440), meta=np.ndarray>
    FI_ispd  (time, nj, ni) float32 5GB dask.array<chunksize=(31, 1080, 1440), meta=np.ndarray>
Attributes:
    sim_name:    LD-fsnow-red
    start_date:  1993-01-01
    end_date:    1994-12-31
    hemisphere:  SH
    ice_type:    FI
    grid_type:   Tc
    method:      raw

[binary-days]
<xarray.Dataset> Size: 10GB
Dimensions:  (time: 729, nj: 1080, ni: 1440)
Coordinates:
  * time     (time) datetime64[ns] 6kB 1993-01-01 1993-01-02 ... 1994-12-30
Dimensions without coordinates: nj, ni
Data variables:
    FI_mask  (time, nj, ni) bool 1GB dask.array<chunksize=(31, 1080, 1440), meta